<a href="https://colab.research.google.com/github/prateekraj91/gpt-learning/blob/main/gpt_learning_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q wandb transformers datasets

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import wandb

import wandb

from datasets import load_dataset
from transformers import AutoTokenizer

import time

In [3]:
dataset = load_dataset(
    "wikitext",
    "wikitext-2-raw-v1"
)

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})


In [4]:
print(dataset["train"][0])

{'text': ''}


In [5]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")

print(tokenizer("hello world"))

{'input_ids': [31373, 995], 'attention_mask': [1, 1]}


In [6]:
def tokenize_function(example):

    return tokenizer(
        example["text"],
        truncation=True,
        max_length=1024
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

In [7]:
# -----------------------------
# Convert dataset into token stream
# -----------------------------

train_ids = []

for example in tokenized_dataset["train"]:

    train_ids.extend(example["input_ids"])

train_ids = torch.tensor(train_ids)

# -----------------------------
# Batch settings
# -----------------------------

batch_size = 1
seq_len = 1024

# -----------------------------
# Batch function
# -----------------------------

def get_batch():

    ix = torch.randint(
        0,
        len(train_ids) - seq_len - 1,
        (batch_size,)
    )

    x = torch.stack([
        train_ids[i:i+seq_len]
        for i in ix
    ])

    y = torch.stack([
        train_ids[i+1:i+seq_len+1]
        for i in ix
    ])

    return x, y

# -----------------------------
# Attention Head
# -----------------------------

class Head(nn.Module):

    def __init__(self, d_model, head_size):

        super().__init__()

        self.key = nn.Linear(d_model, head_size, bias=False)

        self.query = nn.Linear(d_model, head_size, bias=False)

        self.value = nn.Linear(d_model, head_size, bias=False)

        self.register_buffer(
            'mask',
            torch.tril(torch.ones(1024, 1024))
        )

    def forward(self, x):

        B, T, C = x.shape

        k = self.key(x)

        q = self.query(x)

        v = self.value(x)

        scores = q @ k.transpose(-2, -1) / (C ** 0.5)

        scores = scores.masked_fill(
            self.mask[:T, :T] == 0,
            float('-inf')
        )

        weights = F.softmax(scores, dim=-1)

        out = weights @ v

        return out

# -----------------------------
# Multi Head Attention
# -----------------------------

class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, num_heads):

        super().__init__()

        head_size = d_model // num_heads

        self.heads = nn.ModuleList([
            Head(d_model, head_size)
            for _ in range(num_heads)
        ])

        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x):

        out = torch.cat(
            [h(x) for h in self.heads],
            dim=-1
        )

        out = self.proj(out)

        return out

# -----------------------------
# FeedForward
# -----------------------------

class FeedForward(nn.Module):

    def __init__(self, d_model):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.ReLU(),
            nn.Linear(4 * d_model, d_model)
        )

    def forward(self, x):

        return self.net(x)

# -----------------------------
# Transformer Block
# -----------------------------

class Block(nn.Module):

    def __init__(self, d_model, num_heads):

        super().__init__()

        self.attn = MultiHeadAttention(
            d_model,
            num_heads
        )

        self.ffwd = FeedForward(d_model)

        self.ln1 = nn.LayerNorm(d_model)

        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):

        x = x + self.attn(self.ln1(x))

        x = x + self.ffwd(self.ln2(x))

        return x

# -----------------------------
# GPT Model
# -----------------------------

class GPT(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model,
        num_heads,
        num_layers
    ):

        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        self.pos_embedding = nn.Embedding(
            1024,
            d_model
        )

        self.blocks = nn.Sequential(
            *[
                Block(d_model, num_heads)
                for _ in range(num_layers)
            ]
        )

        self.lm_head = nn.Linear(
            d_model,
            vocab_size
        )

    def forward(self, x):

        B, T = x.shape

        token_emb = self.token_embedding(x)

        pos_emb = self.pos_embedding(
            torch.arange(T, device=x.device)
        )

        x = token_emb + pos_emb

        x = self.blocks(x)

        logits = self.lm_head(x)

        return logits

# -----------------------------
# Device
# -----------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------
# Model
# -----------------------------

model = GPT(
    vocab_size=tokenizer.vocab_size,
    d_model=128,
    num_heads=4,
    num_layers=4
).to(device)

# -----------------------------
# Optimizer + Scheduler
# -----------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=200
)

# -----------------------------
# Parameter count
# -----------------------------

print(sum(p.numel() for p in model.parameters()))

# -----------------------------
# wandb
# -----------------------------

wandb.init(
    project="mini-gpt-wikitext"
)
start_time = time.time()

# -----------------------------
# Training loop
# -----------------------------

for step in range(200):

    x, y = get_batch()

    x = x.to(device)

    y = y.to(device)

    logits = model(x)

    B, T, C = logits.shape

    logits = logits.view(B * T, C)

    y = y.view(B * T)

    loss = F.cross_entropy(logits, y)

    ppl = torch.exp(loss)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    scheduler.step()

    wandb.log({
        "train_loss": loss.item(),
        "perplexity": ppl.item(),
        "learning_rate": scheduler.get_last_lr()[0]
    })

    if step % 10 == 0:

      elapsed = time.time() - start_time

      tokens_per_sec = (
        batch_size * seq_len * (step + 1)
    ) / elapsed

    print(
        f"Step {step} | "
        f"Loss: {loss.item():.4f} | "
        f"PPL: {ppl.item():.4f} | "
        f"Tokens/sec: {tokens_per_sec:.2f}"
    )


13838673


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: prateekraj91 (prateekraj91-bits-pil) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step 0 | Loss: 11.2735 | PPL: 78706.1562 | Tokens/sec: 3028.38
Step 1 | Loss: 11.0769 | PPL: 64662.4414 | Tokens/sec: 3028.38
Step 2 | Loss: 11.1121 | PPL: 66979.2891 | Tokens/sec: 3028.38
Step 3 | Loss: 10.9844 | PPL: 58949.8164 | Tokens/sec: 3028.38
Step 4 | Loss: 10.9908 | PPL: 59326.3281 | Tokens/sec: 3028.38
Step 5 | Loss: 10.8126 | PPL: 49642.3320 | Tokens/sec: 3028.38
Step 6 | Loss: 10.8048 | PPL: 49258.2656 | Tokens/sec: 3028.38
Step 7 | Loss: 10.7158 | PPL: 45062.7305 | Tokens/sec: 3028.38
Step 8 | Loss: 10.5718 | PPL: 39017.3633 | Tokens/sec: 3028.38
Step 9 | Loss: 10.5651 | PPL: 38756.4336 | Tokens/sec: 3028.38
Step 10 | Loss: 10.4047 | PPL: 33014.7227 | Tokens/sec: 14129.53
Step 11 | Loss: 10.5551 | PPL: 38371.5234 | Tokens/sec: 14129.53
Step 12 | Loss: 10.3213 | PPL: 30373.0801 | Tokens/sec: 14129.53
Step 13 | Loss: 10.3761 | PPL: 32082.2930 | Tokens/sec: 14129.53
Step 14 | Loss: 10.0501 | PPL: 23157.0039 | Tokens/sec: 14129.53
Step 15 | Loss: 10.1888 | PPL: 26603.2949 | T

In [8]:
val_ids = []

for example in tokenized_dataset["validation"]:

    val_ids.extend(example["input_ids"])

val_ids = torch.tensor(val_ids)

In [9]:
def get_val_batch():

    ix = torch.randint(
        0,
        len(val_ids) - seq_len - 1,
        (batch_size,)
    )

    x = torch.stack([
        val_ids[i:i+seq_len]
        for i in ix
    ])

    y = torch.stack([
        val_ids[i+1:i+seq_len+1]
        for i in ix
    ])

    return x, y

In [10]:
model.eval()

with torch.no_grad():

    x, y = get_val_batch()

    x = x.to(device)

    y = y.to(device)

    logits = model(x)

    B, T, C = logits.shape

    logits = logits.view(B * T, C)

    y = y.view(B * T)

    val_loss = F.cross_entropy(logits, y)

    val_ppl = torch.exp(val_loss)

print("Validation Loss:", val_loss.item())

print("Validation Perplexity:", val_ppl.item())

model.train()

Validation Loss: 7.7141242027282715
Validation Perplexity: 2239.760498046875


GPT(
  (token_embedding): Embedding(50257, 128)
  (pos_embedding): Embedding(1024, 128)
  (blocks): Sequential(
    (0): Block(
      (attn): MultiHeadAttention(
        (heads): ModuleList(
          (0-3): 4 x Head(
            (key): Linear(in_features=128, out_features=32, bias=False)
            (query): Linear(in_features=128, out_features=32, bias=False)
            (value): Linear(in_features=128, out_features=32, bias=False)
          )
        )
        (proj): Linear(in_features=128, out_features=128, bias=True)
      )
      (ffwd): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=128, out_features=512, bias=True)
          (1): ReLU()
          (2): Linear(in_features=512, out_features=128, bias=True)
        )
      )
      (ln1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (ln2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    )
    (1): Block(
      (attn): MultiHeadAttention(
        (heads): ModuleList(
          (0-3

In [11]:
!nvidia-smi

Sat May 23 12:34:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P0             43W /   70W |    1393MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
torch.save(
    model.state_dict(),
    "baseline.pt"
)

In [13]:
!ls

baseline.pt  sample_data  wandb


In [14]:
from google.colab import files

files.download("baseline.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
wandb.finish()

learning_rate,████████▇▇▇▇▇▆▆▆▆▅▅▅▅▅▅▄▄▃▃▃▃▃▃▃▂▂▂▁▁▁▁▁
perplexity,█▆▅▄▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss,██▆▅▅▄▄▄▃▃▃▃▂▂▃▁▂▂▃▁▂▁▂▁▂▃▁▂▁▁▁▁▃▂▁▁▂▁▂▁
learning_rate,0
perplexity,2053.48193
train_loss,7.62729
